In [1]:
import sys
from absl import flags
from ml_collections.config_flags import config_flags

sys.argv = [
    "",
    "--config=td_sa_stack/config.py",
]

config_flags.DEFINE_config_file("config", None, "Training configuration.", lock_config=True)
# flags.DEFINE_string("workdir", None, "Work directory.")
# flags.DEFINE_enum("mode", None, ["train", "eval", "fid_stats"], "Running mode: train, eval or fid_stats")
# flags.DEFINE_string("eval_folder", "eval", "The folder name for storing evaluation results")


FLAGS = flags.FLAGS
FLAGS(sys.argv)

config = FLAGS.config

In [2]:
config.model

activation: swish
name: RegressionInceptionNetV1
optimizer: adamw
optimizer_hparams:
  lr: 0.0001
  weight_decay: 1.0e-05

In [3]:
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU") # Отключение GPU для TensorFlow

import os
import jax
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.8'
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_force_compilation_parallelism=1'

from td_sa_stack import get_dataset, TrainerModule, RegressionInceptionNetV1

2025-06-13 00:21:50.721141: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-13 00:21:50.733503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749774110.745866   19598 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749774110.749919   19598 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749774110.759965   19598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
jax.devices()

[CudaDevice(id=0)]

In [5]:
train_ds, _, _ = get_dataset(config)

trainer = TrainerModule(config=config,
                        model_class=RegressionInceptionNetV1,
                        version=1)

E0613 00:21:56.951281   19598 pjrt_stream_executor_client.cc:3045] Execution of replica 0 failed: UNKNOWN: CUDNN_STATUS_EXECUTION_FAILED
in external/xla/xla/stream_executor/cuda/cuda_dnn.cc(6416): 'status'


XlaRuntimeError: UNKNOWN: CUDNN_STATUS_EXECUTION_FAILED
in external/xla/xla/stream_executor/cuda/cuda_dnn.cc(6416): 'status'

In [6]:
import jax
print(f"JAX version: {jax.__version__}")
print(f"CUDA devices: {jax.devices()}")
print(f"CUDNN version: {jax.lib.xla_bridge.get_backend().platform_version}")


JAX version: 0.5.0
CUDA devices: [CudaDevice(id=0)]
CUDNN version: PJRT C API
cuda 12030


/tmp/ipykernel_19598/3488396981.py:4: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  print(f"CUDNN version: {jax.lib.xla_bridge.get_backend().platform_version}")


In [ ]:
trainer.train_model(train_ds=train_ds)